# **Loading Packages**
_____

In [1]:
import pandas as pd
import numpy as np

import requests
import time
from io import StringIO
import os

from fredapi import Fred
import yfinance as yf

from sklearn.linear_model import LassoCV
from statsmodels.tsa.stattools import adfuller

# **Calling Data**
______

## **FRED**

In [2]:
os.chdir('/Users/dannyhogan/Desktop/ST-498/Code')
with open("API_key.txt", "r") as file:
    api_key = file.read().strip()
fred = Fred(api_key=api_key)

# --- UK FRED Series ---
UK_exchange_rate = fred.get_series('RBGBBIS').resample('QS').first()
UK_oil_price     = fred.get_series('DCOILBRENTEU').resample('QS').first()
UK_bond_yield    = fred.get_series('IRLTLT01GBM156N').resample('QS').first()
UK_GDP           = fred.get_series('NGDPRSAXDCGBQ')
UK_Credit        = fred.get_series('QGBCAMUSDA')

Fred_Data_UK = pd.DataFrame({
    'UK_ExchangeRate': UK_exchange_rate,
    'UK_OilPrice':     UK_oil_price,
    'UK_BondYield':    UK_bond_yield,
    'UK_rGDP':         UK_GDP,
    'UK_Credit':       UK_Credit.pct_change(4) * 100
})

Fred_Data_UK = Fred_Data_UK[Fred_Data_UK.index > '1993-10-01']

# --- US FRED Series ---
US_exchange_rate = fred.get_series('DEXUSEU').resample('QS').first()         # USD/EUR exchange rate
US_oil_price     = fred.get_series('DCOILWTICO').resample('QS').first()      # WTI crude oil
US_bond_yield    = fred.get_series('IRLTLT01USM156N').resample('QS').first() # US 10-year bond yield
US_GDP           = fred.get_series('GDPC1')                                   # US Real GDP (quarterly)
US_Credit        = fred.get_series('DRCCLACBS').resample('QS').first()       # Delinquency Rate on Credit Card Loans

Fred_Data_US = pd.DataFrame({
    'US_ExchangeRate': US_exchange_rate,
    'US_OilPrice':     US_oil_price,
    'US_BondYield':    US_bond_yield,
    'US_rGDP':         US_GDP,
    'US_Credit':       US_Credit  # Already a rate, no pct_change needed
})

Fred_Data_US = Fred_Data_US[Fred_Data_US.index > '1993-10-01']

## **Yahoo Finance**

In [3]:
# --- UK: FTSE 100 ---
ftse = yf.Ticker("^FTSE")
ftse_data = ftse.history(
    start="1993-01-01",
    end=pd.Timestamp.today().strftime('%Y-%m-%d'),
    interval="3mo"
)
ftse_data = ftse_data[['Open']].rename(columns={'Open': 'UK_FTSE_Open'})

# --- US: S&P 500 ---
sp500 = yf.Ticker("^GSPC")
sp500_data = sp500.history(
    start="1993-01-01",
    end=pd.Timestamp.today().strftime('%Y-%m-%d'),
    interval="3mo"
)
sp500_data = sp500_data[['Open']].rename(columns={'Open': 'US_SP500_Open'})

## **OECD**

In [4]:
def fetch_oecd(url, name):
    """Fetch OECD data and return cleaned DataFrame"""
    if 'format=' not in url:
        url += '&format=csvfilewithlabels'
    
    headers = {
        'Accept': 'application/vnd.sdmx.data+csv; charset=utf-8; labels=both',
        'Accept-Encoding': 'gzip, deflate'
    }
    
    time.sleep(1)
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        df = pd.read_csv(StringIO(response.text))
        df_clean = df[['TIME_PERIOD', 'OBS_VALUE']].copy()
        df_clean.columns = ['date', name]
        df_clean[name] = pd.to_numeric(df_clean[name], errors='coerce')
        return df_clean
    else:
        print(f"{name}: Error {response.status_code}")
        return None

# ======================================================================
# Update URLs here
# ======================================================================

# --- UK OECD Queries ---
queries_UK = {
    "UK_house_prices": "https://sdmx.oecd.org/public/rest/data/OECD.ECO.MPD,DSD_AN_HOUSE_PRICES@DF_HOUSE_PRICES,/GBR.Q.RHP.?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions",
    "UK_cpi":          "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_G20_PRICES@DF_G20_PRICES,/GBR.Q...PA...?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions",
    "UK_unemployment": "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_LFS@DF_IALFS_UNE_M,/GBR..._Z.Y._T.Y_GE15..Q?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions"
}

# --- US OECD Queries ---
queries_US = {
    "US_house_prices": "https://sdmx.oecd.org/public/rest/data/OECD.ECO.MPD,DSD_AN_HOUSE_PRICES@DF_HOUSE_PRICES,/USA.Q.RHP.?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions",
    "US_cpi":          "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_G20_PRICES@DF_G20_PRICES,/USA.Q...PA...?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions",
    "US_unemployment": "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_LFS@DF_IALFS_UNE_M,/USA..._Z.Y._T.Y_GE15..Q?startPeriod=1989-Q1&dimensionAtObservation=AllDimensions"
}

dfs_UK = [fetch_oecd(url, name) for name, url in queries_UK.items()]
OECDdata_UK = pd.concat([df.set_index('date') for df in dfs_UK if df is not None], axis=1)

dfs_US = [fetch_oecd(url, name) for name, url in queries_US.items()]
OECDdata_US = pd.concat([df.set_index('date') for df in dfs_US if df is not None], axis=1)

In [5]:
OECDdata_UK

,UK_house_prices,UK_cpi,UK_unemployment
date,,,
1989-Q2,54.135673,5.266667,7.1
1989-Q3,54.293299,5.133333,7.0
1990-Q3,49.250125,7.533333,6.8
1990-Q4,47.990513,7.833333,7.1
1991-Q3,44.559421,7.766667,9.0
...,...,...,...
2020-Q2,110.293465,0.633333,4.1
2022-Q1,120.104977,6.233333,3.8
2022-Q4,116.902087,10.766670,3.9


# **Creating Dataset**
___

In [6]:
# --- UK Dataset ---
OECDdata_UK.index = pd.PeriodIndex(OECDdata_UK.index, freq='Q').to_timestamp()

Fred_Data_UK.index = pd.to_datetime(Fred_Data_UK.index).tz_localize(None)
ftse_data.index    = pd.to_datetime(ftse_data.index).tz_localize(None)

merged_UK = OECDdata_UK.join([Fred_Data_UK, ftse_data], how='outer')
merged_UK = merged_UK[merged_UK.index > '1993-12-31']

# --- US Dataset ---
OECDdata_US.index = pd.PeriodIndex(OECDdata_US.index, freq='Q').to_timestamp()

Fred_Data_US.index = pd.to_datetime(Fred_Data_US.index).tz_localize(None)
sp500_data.index   = pd.to_datetime(sp500_data.index).tz_localize(None)

merged_US = OECDdata_US.join([Fred_Data_US, sp500_data], how='outer')
merged_US = merged_US[merged_US.index > '1993-12-31']

# **Saving Data Locally**
_____

In [7]:
# Be sure to change to your own internal directory
os.chdir('/Users/dannyhogan/Desktop/ST-498')
merged_UK.to_csv("Data/MacroVariablesUK.csv")
merged_US.to_csv("Data/MacroVariablesUS.csv")